|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 1:</h2>|<h1>The Naive Loop<h1>|
|<h2>Section:</h2>|<h1>The KV cache<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: write both loops and make them agree<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
# The naive loop runs a forward pass at every length from 1 to N. So it asks
# the allocator for N different block sizes. The default allocator keeps all
# of them, and then it runs out of room on a card with 12 GB free. This mode
# grows one segment instead. Set it BEFORE you import torch.
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import cudalib

/home/venugopalan/vllm-from-scratch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


You must write two generation loops. Both must produce the same text.

One loop recomputes the whole prefix at every step. The other keeps K and V
and feeds the model one token at a time. Write both. Prove that they agree.
Then find what the cache bought you.

This is a small version of stage 02. It uses HuggingFace's cache instead of
one that you built.

In [2]:
### run this cell

MODEL = 'Qwen/Qwen3-0.6B'
tok   = AutoTokenizer.from_pretrained(MODEL)

def load(dtype):
  return AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype).cuda().eval()

model  = load(torch.bfloat16)
prompt = tok('The capital of France is', return_tensors='pt').input_ids.cuda()
print(tok.decode(prompt[0]))

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 22340.31it/s]

The capital of France is


# Exercise 1: the naive loop

Use greedy decoding and no cache. The whole sequence passes through the model
at every step.

In [3]:
@torch.inference_mode()
def naive_generate(model, ids, n):
  x = ids.clone()
  for _ in range(n):
    logits = model(x, use_cache=False).logits
    nxt    = logits[:, -1:].argmax(-1)
    x      = torch.cat([x, nxt], dim=1)
  return x

out = naive_generate(model, prompt, 20)
print(tok.decode(out[0]))

The capital of France is Paris. The capital of Italy is Rome. The capital of Spain is Madrid. The capital of China


# Exercise 2: the cached loop

The prompt passes through once. After that, each step sees one token. The
model reads the rest from `past_key_values`.

The trap is the input to the second pass. Feed the whole sequence again and
the cache silently appends it to a prefix that it already holds.

In [4]:
@torch.inference_mode()
def cached_generate(model, ids, n):
  # the prompt goes through once, in full. This is prefill.
  out  = model(ids, use_cache=True)
  past = out.past_key_values
  nxt  = out.logits[:, -1:].argmax(-1)
  x    = torch.cat([ids, nxt], dim=1)

  # after that, one token at a time. This is decode.
  for _ in range(n-1):
    out  = model(nxt, past_key_values=past, use_cache=True)
    past = out.past_key_values
    nxt  = out.logits[:, -1:].argmax(-1)
    x    = torch.cat([x, nxt], dim=1)
  return x

out = cached_generate(model, prompt, 20)
print(tok.decode(out[0]))

The capital of France is Paris. The capital of France is also the capital of the Republic of France. The capital of France


# Exercise 3: do they agree?

You use the same model, the same prompt, and the same greedy rule. The two
loops must write the same sentence.

In [5]:
a = naive_generate(model, prompt, 24)
b = cached_generate(model, prompt, 24)

print('identical:', torch.equal(a, b))
print('\nnaive :', tok.decode(a[0]))
print('cached:', tok.decode(b[0]))

diff = next((i for i,(u,v) in enumerate(zip(a[0], b[0])) if u != v), None)
print(f'\nfirst token that differs: {diff}')

identical: False

naive : The capital of France is Paris. The capital of Italy is Rome. The capital of Spain is Madrid. The capital of China is Beijing. The
cached: The capital of France is Paris. The capital of France is also the capital of the Republic of France. The capital of France is also the capital

first token that differs: 10


# Exercise 4: what did the cache buy?

Time both loops at several lengths. One loop is linear in the number of
tokens. The other is not. So the gap must grow as you make more tokens.

The first cell of this notebook sets an allocator option. That option is not
superstition. The naive loop runs a forward pass at 512 different sequence
lengths, so it asks the allocator for 512 different block sizes. The default
allocator keeps all of them. Without the option this notebook reports an
out-of-memory warning on a card with 12 GB free.

Every new shape has a cost. You meet that fact twice more. On the JAX track a
new shape causes a recompile. In stage 12 it is the reason to capture CUDA
graphs at bucketed batch sizes.

In [6]:
import gc

for n in (32, 128, 512):
  gc.collect(); torch.cuda.empty_cache()
  ms_naive  = cudalib.bench_ms(lambda: naive_generate(model, prompt, n),
                               iters=1, warmup=1)
  ms_cached = cudalib.bench_ms(lambda: cached_generate(model, prompt, n),
                               iters=1, warmup=1)
  print(f'{n:>4} tokens: naive {ms_naive/1000:6.2f} s   '
        f'cached {ms_cached/1000:5.2f} s   {ms_naive/ms_cached:5.2f}x')

  32 tokens: naive   0.26 s   cached  0.25 s    1.04x


 128 tokens: naive   1.04 s   cached  0.98 s    1.06x


 512 tokens: naive   6.27 s   cached  3.94 s    1.59x


# Exercise 5: now repeat Exercise 3 in fp32

You use the same two loops and the same prompt. One thing changes. This
exercise runs last, because fp32 weights are twice the size and crowd the
card.

In [7]:
exact  = load(torch.float32)

a = naive_generate(exact, prompt, 24)
b = cached_generate(exact, prompt, 24)
print('fp32 identical:', torch.equal(a, b))

# fp32 is twice the weights. Give the memory back before the timings,
# or the next cell fragments the allocator and thrashes.
import gc
del exact, a, b
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 4238.68it/s]

fp32 identical: True


### Two results, and neither one is a bug

**In bf16 the two loops write different sentences.** They agree for a while
and then separate, usually near the tenth token. In fp32 they stay identical.

Nothing is broken. The cached path reduces in a different order from the
uncached path, so the logits differ by about 1e-2. Where the top two
candidates sit close together, that difference flips an argmax. One different
token then changes every token after it.

This is a real property of low-precision inference, not an artefact of the
exercise. Production LLM serving is therefore not reproducible across batch
sizes or cache settings.

For the same reason, every check in this repo that compares two
implementations runs in fp32. Read the `hf_exact` fixture in
`tests/conftest.py`.

**The speedup is smaller than the complexity argument promised.** At 32 tokens
the cache can even lose. A 0.6B model spends most of a decode step in Python
and in kernel launches. The cache does not help there.

The win grows with the number of tokens, because one side is linear and the
other is not. The win also grows with the model size, because the fixed tax
stays the same.

Now build this properly, with a cache that you own:

    ./vc guide 2